In [1]:
from sentence_transformers import SentenceTransformer

In [2]:
model = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\ashis\Desktop\llm_zoomCAMP_2026\.venv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ashis\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [3]:
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)

In [4]:
d = "You don't need to register. You're accepted. You can also just start learning and submitting homework wothout registering."
dv = model.encode(d)

In [5]:
v1.dot(dv)

np.float32(0.32103553)

In [6]:
q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)

In [7]:
v2.dot(dv)

np.float32(0.02178938)

In [9]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py

'wget' is not recognized as an internal or external command,
operable program or batch file.


In [13]:
import urllib.request

url = "https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py"
urllib.request.urlretrieve(url, "ingest.py")

('ingest.py', <http.client.HTTPMessage at 0x1d5ec4003e0>)

In [14]:
from ingest import load_faq_data

documents = load_faq_data()

In [15]:
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

In [16]:
from tqdm.auto import tqdm

In [17]:
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

1350

In [19]:
import numpy as np
X = np.array(vectors)

In [20]:
X.shape

(1350, 384)

In [21]:
query = "Can I still join the course after the start date?"
v_query = model.encode(query)

In [24]:
scores = X.dot(v_query)
scores

array([ 0.4874058 ,  0.20991942,  0.762941  , ..., -0.08637969,
        0.03759794, -0.03037039], shape=(1350,), dtype=float32)

In [25]:
scores = [v_query.dot(X[i]) for i in range(len(X))]
scores

[np.float32(0.4874058),
 np.float32(0.2099194),
 np.float32(0.762941),
 np.float32(0.44378543),
 np.float32(0.26083994),
 np.float32(0.4866516),
 np.float32(0.3006156),
 np.float32(0.56009996),
 np.float32(0.45960492),
 np.float32(0.25628167),
 np.float32(0.3315327),
 np.float32(0.27318525),
 np.float32(0.27691638),
 np.float32(0.34123003),
 np.float32(0.46007168),
 np.float32(0.2624028),
 np.float32(0.3920008),
 np.float32(0.10854172),
 np.float32(0.27567312),
 np.float32(0.1664682),
 np.float32(0.31437925),
 np.float32(0.006685341),
 np.float32(0.11205027),
 np.float32(0.21905689),
 np.float32(0.3400085),
 np.float32(0.23571292),
 np.float32(0.3271484),
 np.float32(0.1508835),
 np.float32(0.16563289),
 np.float32(0.05545025),
 np.float32(0.2354119),
 np.float32(0.08533012),
 np.float32(-0.0030899234),
 np.float32(-0.042598404),
 np.float32(-0.060277186),
 np.float32(0.006491106),
 np.float32(0.034277488),
 np.float32(-0.04958968),
 np.float32(-0.0006708205),
 np.float32(-0.017013596)

In [26]:
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(2), np.float32(0.762941))

In [27]:
documents[idx]

{'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.",
 'doc_id': '3f1424af17'}

In [28]:
top5 = np.argsort(scores)[-5:]

In [29]:
top5 = top5[::-1]
top5

array([  2, 625, 907, 538,   7])

In [33]:
scores = np.array(scores)

In [34]:
scores[top5]

array([0.762941  , 0.7579372 , 0.7192131 , 0.6536312 , 0.56009996],
      dtype=float32)

In [35]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.762941
{'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.", 'doc_id': '3f1424af17'}

0.7579372
{'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute.", 'doc_id': '2d8b16c2a0'}

0.7192131
{'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related Questions', 

In [36]:
top5

array([  2, 625, 907, 538,   7])

In [39]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=['course'])
vindex.fit(X, documents)

In [40]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vindex.search(query_vector, num_results=5)

In [41]:
results[0]

{'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'doc_id': '74eb249bbf'}

In [42]:
results = vindex.search(
    query_vector,
    filter_dict = {
        "course": "llm-zoomcamp"
    },
    num_results = 5
)

In [43]:
results

[{'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'doc_id': '74eb249bbf'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.',
  'doc_id': '69d122f12e'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the course be offere

In [44]:
results[0]

{'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'doc_id': '74eb249bbf'}

In [45]:
import urllib.request

url = "https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py"
urllib.request.urlretrieve(url, "rag_helper.py")

('rag_helper.py', <http.client.HTTPMessage at 0x1d5ec5b1630>)

In [58]:
from dotenv import load_dotenv
from openai import OpenAI
import os

load_dotenv()
openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [59]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [60]:
from rag_helper import RAGBase

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    model="llama-3.3-70b-versatile"
)

In [61]:
query = "I just found out about the program, can I still sign up?"
assistant.rag(query)

"You can still sign up for the program. However, if you want to receive a certificate, you'll need to submit your project while submissions are still being accepted. Additionally, keep in mind that you'll need to pass the Capstone project to get the certificate, but you can still catch up on any missed homework as it's not mandatory."

In [72]:
class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, qeury, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )
        

In [73]:
vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=openai_client,
)

In [74]:
vector_assistant.rag("the program has already begun, can I still sign up?")

NotFoundError: Error code: 404 - {'error': {'message': 'The model `gpt-5.4-mini` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}}

In [65]:
print(type(model))
print(model)

<class 'sentence_transformers.sentence_transformer.model.SentenceTransformer'>
SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
)


In [66]:
print(hasattr(model, "encode"))

True


In [75]:
class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, qeury, num_results=5):
        
        print(type(self.embedder))
        print(hasattr(self.embedder, "encode"))
        
        query_vector = self.embedder.encode(query)

        print(type(query_vector), len(query_vector))
        
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )
        

In [76]:
vector_assistant.rag("the program has already begun, can I still sign up?")

NotFoundError: Error code: 404 - {'error': {'message': 'The model `gpt-5.4-mini` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}}

In [77]:
RAGVector.search?

Signature: RAGVector.search(self, qeury, num_results=5)
Docstring: <no docstring>
File:      c:\users\ashis\appdata\local\temp\ipykernel_13760\4250690497.py
Type:      function

In [78]:
vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=openai_client,
)

In [79]:
vector_assistant.rag("the program has already begun, can I still sign up?")

<class 'sentence_transformers.sentence_transformer.model.SentenceTransformer'>
True
<class 'numpy.ndarray'> 384


NotFoundError: Error code: 404 - {'error': {'message': 'The model `gpt-5.4-mini` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}}

In [80]:
print(vector_assistant.model)

gpt-5.4-mini


In [81]:
class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, qeury, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )
        

In [82]:
vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=openai_client,
    model="llama-3.3-70b-versatile"
)

In [83]:
vector_assistant.rag("the program has already begun, can I still sign up?")

'Yes, you can still sign up for the program even though it has already begun. According to the provided context, "You can also just start learning and submitting homework (while the form is open) without registering." This suggests that registration is not strictly necessary to participate, and you can join at any time as long as the submission form is still open. However, if you want to receive a certificate, you will need to submit your project while submissions are still being accepted.'

In [85]:
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

In [86]:
vs_index.fit(vectors, documents)

In [89]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vs_index.search(query_vector, num_results=5)

In [90]:
results

[{'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'doc_id': '74eb249bbf'},
 {'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'The course has already started. Can I still join it?',
  'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.',
  'doc_id': '41aabbd7c5'},
 {'course': 'mlops-zoomcamp',
  'section': 'General Cour

In [91]:
results = vs_index.search(
    query_vector,
    filter_dict={
        "course": "llm-zoomcamp"
    },
    num_results=5
)

In [92]:
results

[{'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'doc_id': '74eb249bbf'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.',
  'doc_id': '69d122f12e'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the course be offere

In [93]:
vs_index.close()

In [96]:
vector_assistant.rag('whats queen gambit?')

'The Queen\'s Gambit is not mentioned in the provided context. However, I can tell you that the Queen\'s Gambit is a popular opening move in the game of chess. It involves the white player moving their pawn in front of their queen two spaces forward, and then black responding by capturing the pawn with their own pawn. This opening move is called a "gambit" because white is sacrificing a pawn in order to put pressure on black\'s position and gain a strategic advantage.\n\nThe term "Queen\'s Gambit" has also been used as the title of a popular Netflix series, which is a fictional story about a young girl who becomes a chess prodigy. If you\'re interested in learning more about the Queen\'s Gambit or chess in general, I\'d be happy to help!'